<a href="https://colab.research.google.com/github/AlHartMos/IEU_courses/blob/main/principals_of_programming/PP_fundamentals_names_scope_function_behaviour__class_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scope and Function Behavior

You will learn a mental model that makes Python feel predictable:
- **Names vs objects** (what variables really are)
- **Scope & the LEGB rule** (how Python chooses which name you mean)
- **Sharing, mutation, copying** (why lists “change outside”)
- **Frames & the call stack** (pause/resume during calls)
- **Closures** (functions that remember)
- **Higher-order functions** (passing functions into functions)


# 1) Names vs Objects

A Python “variable” is best understood as:

> a **name** (label) that refers to an **object**.

- A **name** lives in a **namespace** (like the locals of a function).
- An **object** is the thing in memory (an int, a list, a function, …).

Two different names can refer to the same object (this matters a LOT later).

### `==` vs `is`
- `a == b` → do they have the **same value / content**?
- `a is b` → are they the **same object**?

**Identity tip:** `a is b` is equivalent to `id(a) == id(b)` while both objects exist.


In [1]:
# Same contents, different objects
a = [1, 2, 3]
b = [1, 2, 3]

print('a == b:', a == b)
print('a is b:', a is b)
print('id(a) == id(b):', id(a) == id(b))


a == b: True
a is b: False
id(a) == id(b): False


### Rebinding vs mutation (preview)

- **Rebinding**: you make a name refer to a new object.
- **Mutation**: you modify an existing object in place.

Immutable objects (like `int`, `str`, `tuple`) cannot be mutated.
Mutable objects (like `list`) can be mutated.

Watch the `id(...)` values below.


In [2]:
x = 10
print('Before:', x, 'id:', id(x))
x = x + 1
print('After: ', x, 'id:', id(x), '(new int object)')

lst = [10]
print('\nBefore:', lst, 'id:', id(lst))
lst.append(1)
print('After: ', lst, 'id:', id(lst), '(same list object)')


Before: 10 id: 11654664
After:  11 id: 11654696 (new int object)

Before: [10] id: 138859604261248
After:  [10, 1] id: 138859604261248 (same list object)


### Micro-check 1 (predict → run)
1. Will `s1 == s2` be True?
2. Will `s1 is s2` be True?
3. Is it safe to use `is` for strings?

Then run.


In [3]:
s1 = 'hello'
s2 = 'he' + 'llo'
print('s1 == s2:', s1 == s2)
print('s1 is s2:', s1 is s2)
print('ids:', id(s1), id(s2))


s1 == s2: True
s1 is s2: True
ids: 138859993549680 138859993549680


**Takeaway:** you should almost never use `is` for value comparisons.

✅ Good uses of `is`:
- `x is None`
- sentinel objects (special markers)

✅ Use `==` for:
- numbers, strings, lists, tuples, …


# 2) Scope and the LEGB Rule

**Scope** answers: *When Python sees a name like `x`, which `x` does it mean?*

Python uses **LEGB** name lookup order:
1. **L**ocal — inside the current function
2. **E**nclosing — inside outer functions (nested functions)
3. **G**lobal — top level of the module / notebook
4. **B**uiltins — provided by Python everywhere (`len`, `print`, `sum`, …)

Python searches in that order and stops at the first match.

## Important facts
- `if/for/while/try` **do not** create a new scope.
- `def` **does** create a new local scope (a function scope).

## The single most important rule
> If you assign to a name anywhere inside a function, Python treats that name as **local** in that function,
> unless you declare it `global` or `nonlocal`.


## L — Local
Local names are created inside a function call: parameters + assignments.

If a local name has the same spelling as a global name, the local one **shadows** the global.


In [ ]:
x = 100  # global

def demo_local():
    x = 5  # local shadows global
    print('Inside demo_local, x =', x)

demo_local()
print('Outside, x =', x)


## G — Global
Global names live at the top level of the file/notebook.

- You can **read** global names inside a function.
- To **rebind** (assign to) a global name inside a function, you need `global`.


In [ ]:
rate = 0.2

def tax(price):
    return price * rate  # reading global is OK

print(tax(50))


## The classic trap: `UnboundLocalError`

This fails because Python sees an assignment to `count` and decides `count` is local.
So `count + 1` tries to read a *local* variable before it exists.


In [ ]:
count = 0

def inc_bad():
    count = count + 1  # UnboundLocalError

inc_bad()

### Two fixes

**Fix A (recommended):** don’t use globals for normal logic — return values.

**Fix B (rare):** if you truly want to rebind the global name, declare it.


In [ ]:
count = 0

def inc_good(c):
    return c + 1

count = inc_good(count)
print('count =', count)


In [ ]:
count = 0

def inc_global():
    global count
    count = count + 1

inc_global(); inc_global()
print('count =', count)


## B — Builtins and shadowing

Builtins are names like `len`, `sum`, `print`.
You can accidentally overwrite them (shadow them), which creates extremely confusing bugs.


In [ ]:
len_backup = len
len = 42
print('len is now:', len)
len = len_backup
print('len restored:', len('abc'))


## E — Enclosing (preview)
Enclosing scope exists only when you nest functions.
This is where **closures** come from (later).


### Micro-check 2 (LEGB detective)
Predict what each print sees, then run.


In [ ]:
x = 'GLOBAL x'

def outer():
    x = 'ENCLOSING x'
    def inner():
        x = 'LOCAL x'
        print('inner sees:', x)
    inner()
    print('outer sees:', x)

outer()
print('module sees:', x)


# 3) Sharing, Mutation, Copying

Now we focus on **objects** and whether they can change.

We’ll use:
- Immutable: `int`, `str`, `tuple`
- Mutable: `list`

Everything you learn with lists extends to other mutable objects (dicts, sets, most custom classes).

## Aliasing
If two names refer to the same list object, mutating through one name affects the other.


In [ ]:
a = [1, 2, 3]
b = a
print('a is b:', a is b)
b.append(99)
print('a:', a)
print('b:', b)


## Copying
If you want independent lists, you must copy.
For a simple list, slicing is a common shallow copy: `copy = lst[:]`. In general run the method `copy()`: `newlist = list.copy()`


In [ ]:
original = [1, 2, 3]
copy1 = original.copy() # or copy1 = original[:]
copy1.append(7)
print('original:', original)
print('copy1:', copy1)
print('original is copy1:', original is copy1)


## Mutation vs rebinding (the big one)
- `lst.append(...)` mutates the same list.
- `lst = lst + [...]` creates a new list and rebinds the name.

Watch `id(lst)`.


In [ ]:
lst = [0, 1]
print('start:', lst, 'id:', id(lst))
lst.append(2)
print('after mutation:', lst, 'id:', id(lst))
lst = lst + [3]
print('after rebinding:', lst, 'id:', id(lst))


## Passing arguments: “call-by-sharing”

Python passes **object references** into functions.

- The parameter name is local (new name).
- The object may be shared.

If the object is mutable and you mutate it, the caller observes the change.
If you rebind the parameter name, the caller does NOT observe that rebinding.


In [ ]:
def bump_number(n):
    n = n + 1

def bump_list(lst):
    lst.append('NEW')

x = 10
y = [1, 2]

bump_number(x)
bump_list(y)
print('x:', x)
print('y:', y)


### Exercise 1 — Safe sort (don’t mutate input)
Write `safe_sort(nums)` that returns a sorted copy and does not modify the original.


In [ ]:
mylist = [3,5,0,2]
print(mylist)
sortlist = mylist.sort()
print(sortlist)
print(mylist)

In [ ]:
def safe_sort(nums):
    # TODO
    pass

data = [3, 1, 2]
out = safe_sort(data)
print('data:', data)
print('out:', out)
assert data == [3, 1, 2]
assert out == [1, 2, 3]


# 4) Frames and the Call Stack

A **frame** is the local execution context created for a function call.

It contains:
- parameters
- local variables
- where execution should resume after a nested call

When a function calls another function:
1. Python pauses the current frame
2. creates a new frame
3. runs it to `return`
4. deletes it and resumes the previous frame

This stack of frames is the **call stack**.


In [ ]:
def double(n):
    print('  ENTER double, n =', n)
    result = 2 * n
    print('  EXIT  double ->', result)
    return result

def add_then_double(a, b):
    print('ENTER add_then_double, a =', a, 'b =', b)
    s = a + b
    print('  computed s =', s)
    d = double(s)  # pause add_then_double here
    print('EXIT  add_then_double ->', d)
    return d

print('result =', add_then_double(3, 4))


### Micro-check 3 (predict → run)
Predict the print order.


In [ ]:
def A():
    print('A1')
    B()
    print('A2')

def B():
    print('  B1')
    C()
    print('  B2')

def C():
    print('    C1')
    print('    C2')

A()


# 5) Closures

A **closure** is a function that remembers variables from the place where it was created.

This happens when:
- you define a function inside another function
- the inner function uses a variable from the outer function
- and you return the inner function

Closures are a clean alternative to “global settings” and sometimes a lighter alternative to classes.


## Example: a personalized greeter
`make_greeter(title)` returns a function that knows the title forever.


In [ ]:
def make_greeter(title):
    def greet(name):
        return f'Hello, {title} {name}!'
    return greet

greet_dr = make_greeter('Dr.')
greet_captain = make_greeter('Captain')

print(greet_dr('Molina'))
print(greet_captain('Picard'))


## Example: calculating distance from a center

`distance_from_center(center)` returns a function that knows the center forever.

In [ ]:
def distance_from_center(center):
    cx, cy = center  # enclosing variables

    def dist2(point):
        x, y = point
        return ((x - cx)**2 + (y - cy)**2)**0.5

    return dist2

points = [(0, 0), (2, 2), (1, 0), (5, 1)]

distance_from_origen = distance_from_center((0, 0))
distance_from_point = distance_from_center((2, 1))

for point in points:
    print('distance from (0,0): ',distance_from_origen(point),' distance from (2,1): ', distance_from_point(point))


### Using previous exercise to sort points
Try to understand the code below:

In [ ]:
def make_distance_key(center):
    cx, cy = center  # enclosing variables

    def dist2(point):
        x, y = point
        return (x - cx)**2 + (y - cy)**2

    return dist2

points = [(0, 0), (2, 2), (1, 0), (5, 1)]

key_from_origin = make_distance_key((0, 0))
key_from_center = make_distance_key((2, 1))

print(sorted(points, key=key_from_origin))
print(sorted(points, key=key_from_center))

### Exercise 2 — make_power
Write `make_power(k)` returning a function that computes `x**k`.


In [ ]:
def make_power(k):
    # TODO
    pass

square = make_power(2)
cube = make_power(3)
assert square(5) == 25
assert cube(2) == 8


# 6) Higher-Order Functions

Higher-order functions either:
- take a function as input, or
- return a function as output.

You already saw “return a function” in closures.
Now we practice **passing functions into functions**.


## Example: apply a function


In [ ]:
def apply(f, x):
    return f(x)

def square(n):
    return n * n

print(apply(square, 5))
print(apply(abs, -7))


## Example: sorting with `key=`
`sorted(..., key=...)` expects a function that computes a sorting key.


In [ ]:
names = ['Nacho', 'ana', 'Zoe']
print(sorted(names))
print(sorted(names, key=str.lower))


### Exercise 3 — argmax_by
Write `argmax_by(items, score_fn)` that returns the element with the largest score.

Restrictions: do not use `max(..., key=...)`.


In [ ]:
def argmax_by(items, score_fn):
    # TODO
    pass

assert argmax_by(['hi', 'hello', 'yo'], len) == 'hello'
assert argmax_by([3, -10, 5], abs) == -10


### Exercise 4 — compose
Write `compose(f, g)` returning a function `h(x) = f(g(x))`.


In [ ]:
def compose(f, g):
    # TODO
    pass

def shift10(x):
    return x - 10

h = compose(abs, shift10)
assert h(3) == 7
assert h(20) == 10


# 7) Frames and the Call Stack

A **frame** is the local execution context created for a function call.

It contains:
- parameters
- local variables
- where execution should resume after a nested call

When a function calls another function:
1. Python pauses the current frame
2. creates a new frame
3. runs it to `return`
4. deletes it and resumes the previous frame

This stack of frames is the **call stack**.


In [ ]:
def double(n):
    result = 2 * n
    return result

def add_then_double(a, b):
    s = a + b
    d = double(s)  # pause add_then_double here
    return d

print('result =', add_then_double(3, 4))


### Micro-check 3 (predict → run)
Predict the print order.


In [ ]:
def A():
    print('A1')
    B()
    print('A2')

def B():
    print('  B1')
    C()
    print('  B2')

def C():
    print('    C1')
    print('    C2')

A()


# 8) Advanced Exercises (fun + hard)

These are designed so that **understanding beats typing**.
Use contracts + tests.


## Advanced A — Mutation detective (aliasing bug)
This function tries to build a table, but has a classic aliasing bug.
1) Run it.
2) Explain why.
3) Fix it.


In [ ]:
def make_table(rows, cols, fill=0):
    return [[fill] * cols] * rows  # buggy

t = make_table(3, 4, fill=7)
t[0][0] = 99
print(t)


## Advanced B — Build a pipeline (HOF)
Write `pipe(x, funcs)` that applies functions in order.
`pipe(x, [f, g, h]) = h(g(f(x)))`

Extra: return intermediate values too.


In [ ]:
def pipe(x, funcs):
    # TODO
    pass

def add1(n): return n + 1
def double(n): return 2 * n
def square(n): return n * n

assert pipe(3, [add1, double, square]) == 64


## Advanced C — Memoization via closure
Write `make_memoized(f)` returning a cached version of `f`.
Restrict to one integer argument.

Hint: closure + dict in enclosing scope.


In [ ]:
import time

def slow_square(n):
    time.sleep(0.2)
    return n * n

def make_memoized(f):
    # TODO
    pass

fast = make_memoized(slow_square)
t0 = time.time(); fast(10); t1 = time.time();
t2 = time.time(); fast(10); t3 = time.time();
print('first:', round(t1 - t0, 3), 'second:', round(t3 - t2, 3))


---
# Summary
You should now be able to:
- explain `==` vs `is`
- apply LEGB to predict name lookup
- distinguish mutation vs rebinding
- explain nested calls via frames/call stack
- explain closures and the late-binding pitfall
- write higher-order helper functions
